# Class 2 Warm-up — Model → engine → server → gateway

**Teaching arc** (do in order):

| Step | Where | What students learn |
|------|--------|---------------------|
| **1** | **Terminal** | Download GGUF, start **llama.cpp** on `:8081`, `curl` GET/POST |
| **2** | **Docker + browser** | **Open WebUI** chat → your **naive server** on `:8000` |
| **3** | **Modal** | Same load test, **different gateways** under concurrency ([`MODAL.md`](MODAL.md)) |

Then the main lab: [`class2.ipynb`](class2.ipynb) (break naive server, compare A/B/C/D).

**Instructor script** (commands + what to say): [`WARMUP.md`](WARMUP.md)

**Setup** (from `class2/`):

```bash
cd class2
python -m venv .venv && source .venv/bin/activate
pip install -r requirements.txt
export PYTHONPATH=$PWD
```

## Step 1 — llama.cpp engine + HTTP (curl)

**Terminal 1** — downloads GGUF (first run), starts llama.cpp, keeps running:

```bash
cd class2 && source .venv/bin/activate && export PYTHONPATH=$PWD
python inference_101.py
```

**Terminal 2** — copy the printed curls, e.g.:

```bash
curl -s http://127.0.0.1:8081/health | jq
curl -s http://127.0.0.1:8081/v1/chat/completions \
  -H 'Content-Type: application/json' \
  -d '{"model":"llama","messages":[{"role":"user","content":"What is inference?"}],"max_tokens":32}' | jq
```

**Say:** GGUF file on disk → llama.cpp loads it → HTTP API. Same shape ChatGPT clients use.

In [ ]:
!python inference_101.py

## Step 1b — llama.cpp: GGUF file + inference in the console

Different **engine**, same idea — still no long-running server.

```bash
python scripts/download_gguf.py          # once, ~700 MB → models/model.gguf
bash scripts/intro_llama_cli.sh
bash scripts/intro_llama_cli.sh "What is GGUF?"
```

Uses a **one-shot** `docker run` with `llama-cli` — not `docker compose up`.

**Say:** Same TinyLlama family, quantized GGUF, different runtime (llama.cpp vs Transformers).

In [ ]:
# After download_gguf.py in terminal:
!bash scripts/intro_llama_cli.sh "What is an inference engine?"

## Step 2 — Open WebUI + your naive server (Docker)

**Stop Part 1** (Ctrl+C `inference_101.py`). Part 2 adds a **browser chat UI** on top of **your** FastAPI server — not llama.cpp again.

**Terminal 1** — naive server + Open WebUI:

```bash
cd class2
docker compose --profile cpu up
```

Wait until naive-server logs `Ready`. Then open **http://127.0.0.1:3000** in your browser.

1. Pick model **`TinyLlama/TinyLlama-1.1B-Chat-v1.0`** in the dropdown
2. Chat in the UI — requests go to `naive_server/server.py` on `:8000`
3. Optional MCP: **Admin → Settings → External Tools** (add an MCP server URL)

**Terminal 2** — same backend, raw curl (proves UI and curl hit the same API):

```bash
curl -s http://127.0.0.1:8000/healthz | jq
curl -s http://127.0.0.1:8000/v1/chat/completions \
  -H 'Content-Type: application/json' \
  -d '{"messages":[{"role":"user","content":"Hi"}],"max_tokens":8}' | jq
```

**Say:** Part 1 = curl → engine. Part 2 = **client (Open WebUI)** → **your server** → Transformers engine. Code to walk: `naive_server/server.py`.

In [ ]:
import os
import httpx

NAIVE_BASE = os.environ.get("NAIVE_BASE_URL", "http://127.0.0.1:8000")

try:
    print("healthz:", httpx.get(f"{NAIVE_BASE}/healthz", timeout=30).json())
    r = httpx.post(
        f"{NAIVE_BASE}/v1/chat/completions",
        json={"messages": [{"role": "user", "content": "Say hi in five words."}], "max_tokens": 16},
        timeout=120,
    )
    data = r.json()
    print("reply:", data["choices"][0]["message"]["content"].strip())
    print("metrics:", data.get("metrics"))
except httpx.HTTPError as e:
    print("Server not up? Start: docker compose --profile cpu up")
    print(e)

### Step 2b — llama.cpp as an HTTP engine (optional, same API shape)

```bash
docker compose --profile intro up llama-engine-cpu     # Mac, port 8081
# docker compose --profile intro-gpu up llama-engine-gpu
```

```bash
curl -s http://127.0.0.1:8081/health
curl -s http://127.0.0.1:8081/metrics | head
curl -s http://127.0.0.1:8081/v1/chat/completions \
  -H 'Content-Type: application/json' \
  -d '{"model":"llama","messages":[{"role":"user","content":"Hi"}],"max_tokens":8}' | jq
```

In [ ]:
LLAMA_BASE = os.environ.get("LLAMA_BASE_URL", "http://127.0.0.1:8081")

for path in ["/health", "/metrics"]:
    try:
        r = httpx.get(f"{LLAMA_BASE}{path}", timeout=10)
        print(f"{path} -> {r.status_code}")
        print(r.text[:600])
    except httpx.HTTPError as e:
        print(f"{path}: not available ({e})")

## Step 3 — Modal: gateways + concurrency (main class path on Mac)

Deploy four apps, run the **same** break script, switch `TARGET_URL`:

| Sys | App | Role |
|-----|-----|------|
| A | `naive-server` | Transformers — break under load |
| B | `llama-engine` | llama.cpp only — engine swap |
| C | `relay-serve` | RelayServe → same B engine |
| D | `litellm` | LiteLLM → naive (gateway on bad engine) |

```bash
bash scripts/modal.sh deploy all
bash scripts/modal.sh compare

eval "$(bash scripts/modal.sh env naive-server)"
python scripts/part3_break_server.py --base-url "$TARGET_URL"

eval "$(bash scripts/modal.sh env llama-engine)"
python scripts/part3_break_server.py --base-url "$TARGET_URL"

eval "$(bash scripts/modal.sh env relay-serve)"
python scripts/part5_observability.py --url "$METRICS_URL" --interval 1 --count 60
python scripts/part3_break_server.py --base-url "$TARGET_URL"
```

Full write-up: [`MODAL.md`](MODAL.md) · Main lab: [`class2.ipynb`](class2.ipynb)

### One-line recap

1. **Console** — you own model + engine  
2. **Docker HTTP** — server exposes chat API (ChatGPT-shaped)  
3. **Modal gateways** — who handles concurrency when load spikes?